In [2]:
import pickle

import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

from tensorflow import keras
import kerasncp as kncp

import os
from typing import Iterable, Dict
import tensorflow as tf
import kerasncp as kncp
from kerasncp.tf import LTCCell, WiredCfcCell
from tensorflow import keras
import numpy as np
from matplotlib.image import imread
from tqdm import tqdm
from PIL import Image
import pandas as pd
import time
from keras_models import generate_ncp_model


In [3]:
training_root = "../../fly_to_target_dataset/diff_coreset"
val_root = "../../fly_to_target_dataset/test_data"
DROPOUT = 0.1

DEFAULT_NCP_SEED = 22222

IMAGE_SHAPE = (144, 256, 3)
IMAGE_SHAPE_CV = (IMAGE_SHAPE[1], IMAGE_SHAPE[0])

# goal_image_pth = "goal_image.png"
# goal_image = imread(goal_image_pth)
# goal_image = np.array(goal_image)
# goal_image = goal_image / 255.0

In [4]:
!export TF_CPP_MIN_LOG_LEVEL=2

In [ ]:
# batch_size = None
# seq_len = 64
# augmentation_params = None
# single_step = False
# no_norm_layer = False

# decay_rate: float = 0.85
# lr: float = 0.001
# lr_schedule = keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=lr, decay_steps=500,
#                                                             decay_rate=decay_rate, staircase=True)
# #Adam optimizer
# optimizer = keras.optimizers.Adam(learning_rate=lr_schedule)

# gpus = tf.config.list_logical_devices('GPU')
# strategy = tf.distribute.MirroredStrategy(gpus)
# with strategy.scope():
#     mymodel = generate_ncp_model(seq_len, IMAGE_SHAPE, augmentation_params, batch_size, DEFAULT_NCP_SEED, single_step, no_norm_layer)
#     mymodel.compile(optimizer=optimizer, loss='mse', metrics=['mse'])
#     mymodel.load_weights('../saved_models/fine_tuned_woscheduler_seed22222_lr0.0001_trainloss0.00012_valloss0.08719_diff_dataset.h5')

#     mymodel.summary()

In [5]:
def get_output_normalization(root):
    training_output_mean_fn = os.path.join(root, 'stats', 'training_output_means.csv')
    if os.path.exists(training_output_mean_fn):
        print('Loading training data output means from: %s' % training_output_mean_fn)
        output_means = np.genfromtxt(training_output_mean_fn, delimiter=',')
    else:
        output_means = np.zeros(4)

    training_output_std_fn = os.path.join(root, 'stats', 'training_output_stds.csv')
    if os.path.exists(training_output_std_fn):
        print('Loading training data output std from: %s' % training_output_std_fn)
        output_stds = np.genfromtxt(training_output_std_fn, delimiter=',')
    else:
        output_stds = np.ones(4)

    return output_means, output_stds


def load_dataset_multi(root, image_size, seq_len, shift, stride, label_scale):
    file_ending = 'png'
    IMAGE_SHAPE = (144, 256, 3)
    IMAGE_SHAPE_CV = (IMAGE_SHAPE[1], IMAGE_SHAPE[0])

    def sub_to_batch(sub_feature, sub_label):
        sfb = sub_feature.batch(seq_len, drop_remainder=True)
        slb = sub_label.batch(seq_len, drop_remainder=True)
        return tf.data.Dataset.zip((sfb, slb))
        # return sub.batch(seq_len, drop_remainder=True)
    
    def apply_random_augmentations(image):
        # Generate a random number and apply augmentations with a 50% probability
        if tf.random.uniform(()) > 0.3:  # 30% chance to apply augmentations
            image = tf.image.convert_image_dtype(image, tf.float32)  # Convert to float32 for augmentation
            image = tf.image.random_brightness(image, max_delta=0.1)  # Random brightness adjustment
            image = tf.image.random_contrast(image, lower=0.8, upper=1.2)  # Random contrast adjustment
            image = tf.image.random_saturation(image, lower=0.8, upper=1.2)  # Random saturation adjustment
            image = tf.image.convert_image_dtype(image, tf.uint8)  # Convert back to uint8
        return image

    
    datasets = []

    #output_means, output_stds = get_output_normalization(root)

    
    for i in range(len(os.listdir(root)))[:1]:
        directory = i + 1
        csv_file_name = f"{root}/{str(directory)}/data_out.csv"
        labels = np.genfromtxt(csv_file_name, delimiter=',', skip_header=1, dtype=np.float32)
        print("labels", labels)
        # if labels.shape[1] == 4:
        #     labels = (labels - output_means) / output_stds
        #     # labels = labels * label_scale
        # elif labels.shape[1] == 5:
        #     labels = (labels[:, 1:] - output_means) / output_stds
        #     # labels = labels[:,1:] * label_scale
        # else:
        #     raise Exception('Wrong size of input data (expected 4, got %d' % labels.shape[1])
    
        labels_dataset = tf.data.Dataset.from_tensor_slices(labels)
        # n_images = len(os.listdir(os.path.join(root, d))) - 1
        n_images = len([fn for fn in os.listdir(f"./{root}/{str(directory)}") if file_ending in fn])
        print(n_images)
        print("no of imgs", n_images)
        # dataset_np = np.empty((n_images, 256, 256, 3), dtype=np.uint8)
        dataset_np = np.empty((n_images, *image_size), dtype=np.uint8)

        for ix in range(n_images):
            # dataset_np[ix] = imread(os.path.join(root, d, '%06d.jpeg' % ix))
            img_file_name = root + "/" + str(directory) +'/Image' + str(ix + 1) + '.'+ file_ending
            img = Image.open(img_file_name)
            img = img.resize(IMAGE_SHAPE_CV)
            # dataset_np[ix] = img[img.height - image_size[0]:, :, :]
            dataset_np[ix] = img

        images_dataset = tf.data.Dataset.from_tensor_slices(dataset_np)
        images_dataset = images_dataset.map(apply_random_augmentations, num_parallel_calls=tf.data.experimental.AUTOTUNE)
        dataset = tf.data.Dataset.zip((images_dataset, labels_dataset))
        dataset = dataset.window(seq_len, shift=shift, stride=stride, drop_remainder=True).flat_map(sub_to_batch)
        datasets.append(dataset)

    return datasets

def get_dataset_multi(root, image_size, seq_len, shift, stride, validation_ratio, label_scale, extra_data_root=None):
    ds = load_dataset_multi(root, image_size, seq_len, shift, stride, label_scale)
    print('n bags: %d' % len(ds))
    cnt = 0

    for d in ds:
        for (ix, _) in enumerate(d):
            pass
            cnt += ix
    print('n windows: %d' % cnt)

    val_ix = 0

    # val_ix = int(len(ds) * validation_ratio)
    # print('\nval_ix: %d\n' % val_ix)
    # validation_datasets = ds[:val_ix]

    training_datasets = ds[val_ix:]

    # if either dataset has length 0, trying to call flat map raises error that return type is wrong
    # assert len(training_datasets) > 0 and len(validation_datasets) > 0, f"Training or validation dataset has no points!" \
    #                                                                     f"Train dataset len: {len(training_datasets)}" \
    #                                                                     f"Val dataset len: {len(validation_datasets)}"
    training = tf.data.Dataset.from_tensor_slices(training_datasets).flat_map(lambda x: x)
    # validation = tf.data.Dataset.from_tensor_slices(validation_datasets).flat_map(lambda x: x)

    # return training, validation
    return training


In [6]:
shift: int = 1
stride: int = 1
# decay_rate: float = 0.95
val_split: float = 0.2
label_scale: float = 1
seq_len = 64
val_split: float = 0.1
label_scale: float = 1

with tf.device('/cpu:0'):
    training_dataset = get_dataset_multi(training_root, IMAGE_SHAPE, seq_len, shift, stride, val_split, label_scale, extra_data_root=None)
    

2024-11-13 15:55:32.271658: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2024-11-13 15:55:32.272987: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2024-11-13 15:55:32.296866: E tensorflow/stream_executor/cuda/cuda_driver.cc:328] failed call to cuInit: CUDA_ERROR_NO_DEVICE: no CUDA-capable device is detected
2024-11-13 15:55:32.296911: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:169] retrieving CUDA diagnostic information for host: iras-hub
2024-11-13 15:55:32.296927: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:176] hostname: iras-hub
2024-11-13 15:55:32.297080: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:200] libcuda reported version is: 550.54.14
2024-11-13 15:55:32.297117: I tensorflow/stream_executor/cuda/cuda_diagnostics.cc:204] kernel reported version is: 550.54.14
2024-11-13 15:55:32.297128: I tensorflow/stream_executor/cu

labels [[-5.45208991e-01  5.56323528e-01 -9.62745726e-01  5.51216781e-01]
 [-4.69735712e-01  5.59266984e-01 -8.87602270e-01  5.16884804e-01]
 [-3.96264791e-01  5.48152566e-01 -8.28002930e-01  4.87919092e-01]
 [-3.18426251e-01  5.24390161e-01 -7.62720525e-01  4.55274433e-01]
 [-2.60237217e-01  4.96815681e-01 -7.08253622e-01  4.27806526e-01]
 [-2.19786286e-01  4.72018570e-01 -6.70199931e-01  4.07966048e-01]
 [-1.71101779e-01  4.34729010e-01 -6.20564520e-01  3.81809920e-01]
 [-1.26528889e-01  3.93339336e-01 -5.77598214e-01  3.56382489e-01]
 [-9.09503549e-02  3.50732327e-01 -5.36118031e-01  3.34085882e-01]
 [-7.03007281e-02  3.23684692e-01 -5.10422289e-01  3.17597598e-01]
 [-4.33358178e-02  2.80392528e-01 -4.69446898e-01  2.94971615e-01]
 [-2.49505118e-02  2.43477389e-01 -4.32806820e-01  2.73863882e-01]
 [-1.16705829e-02  2.13092744e-01 -4.04083222e-01  2.56695092e-01]
 [-6.24483393e-04  1.82767749e-01 -3.74043435e-01  2.38524824e-01]
 [ 4.71147848e-03  1.65640324e-01 -3.56202573e-01  2.27

2024-11-13 15:55:33.049835: I tensorflow/compiler/mlir/mlir_graph_optimization_pass.cc:116] None of the MLIR optimization passes are enabled (registered 2)
2024-11-13 15:55:33.068565: I tensorflow/core/platform/profile_utils/cpu_utils.cc:112] CPU Frequency: 3000260000 Hz


In [7]:
print('load dataset shape', training_dataset.element_spec)
training_dataset = training_dataset.shuffle(100).batch(64)


load dataset shape (TensorSpec(shape=(64, 144, 256, 3), dtype=tf.uint8, name=None), TensorSpec(shape=(64, 4), dtype=tf.float32, name=None))


In [ ]:
# class PrintInputCallback(tf.keras.callbacks.Callback):
#     def on_train_batch_begin(self, batch, logs=None):
#         # Print the batch number to keep track
#         print(f"\nBatch {batch}:")

        
#         inputs, targets = next(iter(training_dataset))
#         tf.print("Input data for batch:", inputs.shape, targets.shape)

In [ ]:
# options = tf.data.Options()
# options.experimental_distribute.auto_shard_policy = tf.data.experimental.AutoShardPolicy.DATA
# training_dataset = training_dataset.with_options(options)

# # Have GPU prefetch next training batch while first one runs
# training_dataset = training_dataset.prefetch(tf.data.AUTOTUNE)



# epochs: int = 1

# #setting validation data to None
# history = mymodel.fit(x=training_dataset, epochs=epochs,verbose=1, use_multiprocessing=False, workers=1, max_queue_size=5, callbacks=[PrintInputCallback()])
# print(history)

# # Extract the final training and validation loss
# train_loss = history.history['loss'][-1]
# # val_loss = history.history['val_loss'][-1]


# accuracy = mymodel.evaluate(x=training_dataset)
# print('Accuracy:' ,accuracy)

In [8]:
batch_size = None
seq_len = 64
augmentation_params = None
single_step = False
no_norm_layer = False

image_pth = "goal_img_diff.png"
img = Image.open(image_pth)
img = img.resize(IMAGE_SHAPE_CV)  
img_array = np.array(img) / 255.0
img_arrays = np.stack([img_array] * seq_len, axis = 0)
img_arrays = np.expand_dims(img_arrays, axis = 0) 
goal_image = tf.convert_to_tensor(img_arrays, dtype=tf.float32) 

In [12]:
mymodel = generate_ncp_model(seq_len, IMAGE_SHAPE, augmentation_params, batch_size, DEFAULT_NCP_SEED, single_step, no_norm_layer)

lr: float = 0.0001
#Adam optimizer
optimizer = keras.optimizers.Adam(learning_rate=lr)

# Custom training loop
epochs = 10

losses = []
for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")

    # Iterate over the training dataset
    for step, (x_batch_train, y_batch_train) in enumerate(training_dataset):
        print(f"Step {step}, x_batch_train shape: {x_batch_train.shape}, y_batch_train shape: {y_batch_train.shape}")
        with tf.GradientTape() as tape:
            # Forward pass
            y_pred = mymodel(x_batch_train, training=True)

            # Compute the loss between predictions and targets
            loss1 = tf.reduce_mean(tf.keras.losses.mean_squared_error(y_batch_train, y_pred))

            squared_diff = tf.square(tf.cast(x_batch_train, tf.float32) - goal_image)
            mse_per_image = tf.reduce_mean(squared_diff, axis=[2, 3, 4])
            loss2 = tf.reduce_mean(tf.reduce_sum(mse_per_image, axis=1))            

            # Combine the losses 
            total_loss = loss1 + 0.0000001 * loss2 
            
        # Compute gradients
        gradients = tape.gradient(total_loss, mymodel.trainable_weights)

        # Update weights
        optimizer.apply_gradients(zip(gradients, mymodel.trainable_weights))

        print(f"Step {step}, Loss: {total_loss}")
        losses.append(total_loss)





Epoch 1/10
Step 0, x_batch_train shape: (1, 64, 144, 256, 3), y_batch_train shape: (1, 64, 4)
Step 0, Loss: 0.3447715938091278

Epoch 2/10
Step 0, x_batch_train shape: (1, 64, 144, 256, 3), y_batch_train shape: (1, 64, 4)
Step 0, Loss: 0.27567270398139954

Epoch 3/10
Step 0, x_batch_train shape: (1, 64, 144, 256, 3), y_batch_train shape: (1, 64, 4)
Step 0, Loss: 0.2207806259393692

Epoch 4/10
Step 0, x_batch_train shape: (1, 64, 144, 256, 3), y_batch_train shape: (1, 64, 4)
Step 0, Loss: 0.17671814560890198

Epoch 5/10
Step 0, x_batch_train shape: (1, 64, 144, 256, 3), y_batch_train shape: (1, 64, 4)
Step 0, Loss: 0.20408858358860016

Epoch 6/10
Step 0, x_batch_train shape: (1, 64, 144, 256, 3), y_batch_train shape: (1, 64, 4)
Step 0, Loss: 0.19897949695587158

Epoch 7/10
Step 0, x_batch_train shape: (1, 64, 144, 256, 3), y_batch_train shape: (1, 64, 4)
Step 0, Loss: 0.1951807290315628

Epoch 8/10
Step 0, x_batch_train shape: (1, 64, 144, 256, 3), y_batch_train shape: (1, 64, 4)
Step 